# Basics: containers, norms and products

The first programs of MTL5's tutorial build vectors and matrices, take their
norms, and multiply them, dense and sparse. This notebook does the same from
Python. One thing is new: an mtl5 container is not a copy of your data. It
wraps the NumPy array's memory.

**Uses:** `mtl5.vector`, `mtl5.matrix`, `mtl5.norm`, `mtl5.dot`, `mtl5.matvec`,
`mtl5.matmul`, `mtl5.sparse.coo_matrix`.

**From C++:** [`intro/hello_mtl5.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/intro/hello_mtl5.cpp), [`phase01_core_types/vectors_and_matrices.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase01_core_types/vectors_and_matrices.cpp), [`phase02_basic_operations/norms_and_products.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase02_basic_operations/norms_and_products.cpp),
[`phase02_basic_operations/matrix_arithmetic.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase02_basic_operations/matrix_arithmetic.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## Containers over NumPy memory

`mtl5.vector(a)` and `mtl5.matrix(a)` don't copy: the container and the array
are one buffer, so a write through either one shows in both.

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
v = mtl5.vector(a)

a[0] = 10.0  # write through NumPy ...
print(v.to_numpy())  # ... and the mtl5 vector sees it

assert v.to_numpy()[0] == 10.0
assert np.shares_memory(np.asarray(v), a)

Because nothing is copied, the array has to be in a layout and a type mtl5
can use as it is. A strided slice, or a dtype mtl5 has no container for, raises
`TypeError`. Nothing is converted silently. To convert on purpose, call
`np.ascontiguousarray` or `astype` first.

In [ ]:
for name, arr in [("every other element", a[::2]), ("float16", a.astype(np.float16))]:
    try:
        mtl5.vector(arr)
    except TypeError:
        print(f"{name}: TypeError, not copied")
    else:
        raise AssertionError(f"{name} should not be accepted")

w = mtl5.vector(np.ascontiguousarray(a[::2]))  # an explicit copy is fine
print(w.to_numpy())

## Norms

The tutorial's vector is v = (3, −4, 0, 5, −2), so ‖v‖₁ = 14, ‖v‖₂ = √54 and
‖v‖∞ = 5. `mtl5.norm` takes `ord=1`, `2` (the default) or `-1` for the
infinity norm.

In [ ]:
v = mtl5.vector(np.array([3.0, -4.0, 0.0, 5.0, -2.0]))
one, two, inf = mtl5.norm(v, 1), mtl5.norm(v), mtl5.norm(v, -1)
print(f"one norm {one}, two norm {two:.6f}, infinity norm {inf}")

assert one == 14.0 and inf == 5.0
assert np.isclose(two, np.sqrt(54.0), rtol=1e-15)

Each norm has its own idea of length. Scale every direction in the plane to
length 1 and the three norms draw three unit balls: a diamond, a circle and a
square.

In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 721)
directions = np.column_stack([np.cos(theta), np.sin(theta)])

fig, ax = plt.subplots(figsize=(4.2, 4.2))
for color, (ord_, label) in zip(SERIES, [(1, "1-norm"), (2, "2-norm"), (-1, "∞-norm")]):
    ball = np.array([d / mtl5.norm(d, ord_) for d in directions])
    ax.plot(ball[:, 0], ball[:, 1], color=color, label=label)
    # every point on the curve has length 1 in its own norm
    assert np.allclose([mtl5.norm(p, ord_) for p in ball], 1.0)
ax.set_aspect("equal")
ax.set_title("Unit balls of three norms")
ax.legend(frameon=False, loc="center")
ax.grid(True)
plt.show()

## Products

The dot product, a matrix–vector product and a matrix–matrix product, with
the tutorial's numbers. Multiplying by a vector of ones sums each row.
`mtl5.matmul(P, Q)` and `P @ Q` are the same product.

In [ ]:
x, y = mtl5.vector(np.array([1.0, 2.0, 3.0])), mtl5.vector(np.array([4.0, 5.0, 6.0]))
print("dot =", mtl5.dot(x, y))  # 1*4 + 2*5 + 3*6

A = mtl5.matrix(np.arange(1.0, 10.0).reshape(3, 3))
row_sums = mtl5.matvec(A, mtl5.vector(np.ones(3))).to_numpy()
print("A @ ones =", row_sums)

P = mtl5.matrix(np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
Q = mtl5.matrix(np.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]]))
PQ = (P @ Q).to_numpy()
print("P @ Q =\n", PQ)

assert mtl5.dot(x, y) == 32.0
assert np.array_equal(row_sums, [6.0, 15.0, 24.0])
assert np.array_equal(PQ, [[58.0, 64.0], [139.0, 154.0]])
assert np.array_equal(mtl5.matmul(P, Q).to_numpy(), PQ)

Everything NumPy does to an array also works on the buffer under an mtl5
matrix, because it is one buffer. MTL5's `trace` and `diagonal` are `np.trace`
and `np.diag` here.

In [ ]:
D = mtl5.matrix(np.diag([1.0, 2.0, 3.0, 4.0]))
print("trace =", np.trace(D.to_numpy()), " diagonal =", np.diag(D.to_numpy()))
assert np.trace(D.to_numpy()) == 10.0

## Dense and sparse storage, same product

The C++ tutorial builds a sparse tridiagonal matrix with an inserter. The
Python counterpart is a COO matrix that takes one entry at a time with
`insert`. An entry inserted twice adds up, as finite-element assembly needs.
`tocsr()` then gives compressed rows. Its product with a vector is the dense
product exactly.

In [ ]:
def tridiagonal(n, diag=4.0, off=-1.0):
    T = ms.coo_matrix(np.array([], np.int64), np.array([], np.int64), np.array([]), (n, n))
    for i in range(n):
        T.insert(i, i, diag)
        if i + 1 < n:
            T.insert(i, i + 1, off)
            T.insert(i + 1, i, off)
    return T.tocsr()


S = tridiagonal(5)
dense = mtl5.matrix(
    np.diag(np.full(5, 4.0)) + np.diag(np.full(4, -1.0), 1) + np.diag(np.full(4, -1.0), -1)
)
xs = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

y_sparse = S.matvec(xs).to_numpy()
y_dense = mtl5.matvec(dense, mtl5.vector(xs)).to_numpy()
print(f"nnz = {S.nnz}\nsparse: {y_sparse}\ndense:  {y_dense}")
assert S.nnz == 13
assert np.array_equal(y_sparse, y_dense)

The transposed product needs no transposed copy: `rmatvec` computes Rᵀw from R.
With w all ones, it sums each column.

In [ ]:
R = ms.coo_matrix(
    np.array([0, 0, 1, 1, 2, 2]),
    np.array([0, 1, 0, 1, 0, 1]),
    np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0]),
    (3, 2),
).tocsr()
col_sums = R.rmatvec(np.ones(3)).to_numpy()
print("R^T @ ones =", col_sums)
assert np.array_equal(col_sums, [9.0, 12.0])

The two storages differ in memory. A dense n × n matrix stores n² values.
Compressed rows store each nonzero with its column index, plus one row pointer
per row. For a tridiagonal matrix, that is about 3n of each.

In [ ]:
sizes = np.array([10, 30, 100, 300, 1000, 3000])
dense_bytes = 8.0 * sizes**2
csr_bytes = np.array([sum(arr.nbytes for arr in tridiagonal(n).to_csr_arrays()) for n in sizes])

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.loglog(sizes, dense_bytes / 2**20, "o-", color=SERIES[0], label="dense")
ax.loglog(sizes, csr_bytes / 2**20, "s-", color=SERIES[1], label="compressed rows")
ax.set_xlabel("n")
ax.set_ylabel("memory (MiB)")
ax.set_title("Tridiagonal n × n matrix: memory by storage")
ax.legend(frameon=False)
ax.grid(True, which="major")
plt.show()

# CSR grows linearly: 10x the rows, 10x the bytes, give or take the row pointers.
assert np.isclose(csr_bytes[-1] / csr_bytes[-3], 10.0, rtol=0.01)
assert (csr_bytes < dense_bytes).all()

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `vec::dense_vector<double> v = {...}` | `mtl5.vector(np.array([...]))`, over the array's memory |
| `one_norm(v)`, `two_norm(v)`, `infinity_norm(v)` | `mtl5.norm(v, 1)`, `mtl5.norm(v)`, `mtl5.norm(v, -1)` |
| `dot(a, b)` | `mtl5.dot(a, b)` |
| `A * x` | `mtl5.matvec(A, x)`, or `S.matvec(x)` for a sparse `S` |
| `mult(P, Q, PQ)` | `mtl5.matmul(P, Q)` or `P @ Q` |
| `trans(R) * w` | `R.rmatvec(w)` |
| `inserter<compressed2D<double>> ins(S); ins[i][j] << v;` | `T = mtl5.sparse.coo_matrix(...)`, `T.insert(i, j, v)`, `T.tocsr()` |
| `trace(D)`, `diagonal(D)` | `np.trace`, `np.diag` on the same buffer |

The C++ programs also check concepts such as `Field<double>` and show fixed
compile-time dimensions. Those are compile-time features with no Python
counterpart, so this notebook leaves them out.